In [ ]:
#The code uses UCI HAR Dataset (train + test).

#Sensors: Accelerometer + Gyroscope (x, y, z axes).

#Classes: 6 activities – WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING.

#Problem type: Multiclass classification.

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB 
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import joblib

In [3]:
DATA_DIR = './UCI HAR DATASET/UCI HAR Dataset'

In [18]:
print("Dataset: UCI HAR Dataset")
print("Sensors: Accelerometer + Gyroscope (x,y,z axes)")
print("Activities (classes): WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING")
print("Dataset source: https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones")
print("Total number of samples (windows):", len(y))

Dataset: UCI HAR Dataset
Sensors: Accelerometer + Gyroscope (x,y,z axes)
Activities (classes): WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING
Dataset source: https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones
Total number of samples (windows): 10299


In [5]:
def load_signals(path, split='train'):
    signals = []
    axes = ['x', 'y', 'z']
    for sensor in ['body_acc', 'body_gyro']:
        for axis in axes:
            file_path = os.path.join(path, split, 'Inertial Signals', f'{sensor}_{axis}_{split}.txt')
            data = pd.read_csv(file_path, sep='\\s+', header=None)
            signals.append(data)
    return np.stack(signals, axis=2)

In [6]:
X_train_raw = load_signals(DATA_DIR, 'train')
X_test_raw = load_signals(DATA_DIR, 'test')

y_train = pd.read_csv(os.path.join(DATA_DIR, 'train', 'y_train.txt'), header=None)[0]
y_test = pd.read_csv(os.path.join(DATA_DIR, 'test', 'y_test.txt'), header=None)[0]

subjects_train = pd.read_csv(os.path.join(DATA_DIR, 'train', 'subject_train.txt'), header=None)[0]
subjects_test = pd.read_csv(os.path.join(DATA_DIR, 'test', 'subject_test.txt'), header=None)[0]


In [7]:
X_raw = np.vstack([X_train_raw, X_test_raw])
y = pd.concat([y_train, y_test]).values
subjects = pd.concat([subjects_train, subjects_test]).values

In [8]:
activity_map = {1: 'WALKING', 2: 'WALKING_UPSTAIRS', 3: 'WALKING_DOWNSTAIRS',
                4: 'SITTING', 5: 'STANDING', 6: 'LAYING'}
y_named = np.vectorize(activity_map.get)(y)


In [9]:
def stat_features(signal):
    return [signal.mean(), signal.std(), signal.min(), signal.max(), stats.skew(signal), stats.kurtosis(signal)]

def magnitude(x, y, z):
    return np.sqrt(x**2 + y**2 + z**2)
def sma(signal):
    return np.sum(np.abs(signal)) / len(signal)
def energy(signal):
    return np.sum(signal**2) / len(signal)      


In [19]:
features = []
for i, window in enumerate(X_raw):

    if i % 500 == 0:
        print(f"Processing window {i}/{len(X_raw)}")

    row = []
    acc = window[:, 0:3]
    gyro = window[:, 3:6]

    # Statistical features
    for axis in range(3):
        row.extend(stat_features(acc[:, axis]))
    for axis in range(3):
        row.extend(stat_features(gyro[:, axis]))

    # Magnitude features
    acc_mag = magnitude(*acc.T)
    gyro_mag = magnitude(*gyro.T)
    row.extend(stat_features(acc_mag))
    row.extend(stat_features(gyro_mag))

    # Frequency-domain feature: dominant frequency
    def dominant_freq(signal):
        sp = np.abs(np.fft.fft(signal))
        return np.argmax(sp[1:]) + 1  

    row.append(dominant_freq(acc_mag))
    row.append(dominant_freq(gyro_mag))

    # SMA features
    for axis in range(3):
        row.append(sma(acc[:, axis]))
    for axis in range(3):
        row.append(sma(gyro[:, axis]))
    row.append(sma(acc_mag))
    row.append(sma(gyro_mag))

    # Energy features
    for axis in range(3):
        row.append(energy(acc[:, axis]))
    for axis in range(3):
        row.append(energy(gyro[:, axis]))
    row.append(energy(acc_mag))
    row.append(energy(gyro_mag))

    features.append(row)

# Create dataframe the loop
features_df = pd.DataFrame(features)
features_df['Activity'] = y_named
features_df['Subject'] = subjects

# Save extracted features
features_df.to_csv('features.csv', index=False)
print("Saved features.csv")


Processing window 0/10299
Processing window 500/10299
Processing window 1000/10299
Processing window 1500/10299
Processing window 2000/10299
Processing window 2500/10299
Processing window 3000/10299
Processing window 3500/10299
Processing window 4000/10299
Processing window 4500/10299
Processing window 5000/10299
Processing window 5500/10299
Processing window 6000/10299
Processing window 6500/10299
Processing window 7000/10299
Processing window 7500/10299
Processing window 8000/10299
Processing window 8500/10299
Processing window 9000/10299
Processing window 9500/10299
Processing window 10000/10299
Saved features.csv


In [11]:
#Prepare data fr ML
X = features_df.drop(columns=['Activity', 'Subject'])
y_labels = features_df['Activity']
X_train, X_val, y_train, y_val = train_test_split(X, y_labels, test_size=0.2, random_state=42, stratify=y_labels)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)


In [12]:
#Train multiple models

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVC': SVC(probability=True),
    'RandomForest': RandomForestClassifier(n_estimators=100),
    'NaiveBayes': GaussianNB()
}
results = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred, average='weighted')
    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'y_pred': y_pred, 'model': clf}
    print(f"{name}: acc={acc:.4f}, precision={prec:.4f}, recall={rec:.4f}, f1={f1:.4f}")

LogisticRegression: acc=0.8228, precision=0.8249, recall=0.8228, f1=0.8230
SVC: acc=0.7791, precision=0.7814, recall=0.7791, f1=0.7789
RandomForest: acc=0.9121, precision=0.9128, recall=0.9121, f1=0.9123
NaiveBayes: acc=0.5515, precision=0.6105, recall=0.5515, f1=0.4966


In [14]:
#Confusion matrix
best_model_name = max(results.keys(), key=lambda k: results[k]['f1'])
best = results[best_model_name]
cm = confusion_matrix(y_val, best['y_pred'], labels=list(activity_map.values()))
cm_df = pd.DataFrame(cm, index=list(activity_map.values()), columns=list(activity_map.values()))
cm_df.to_csv('confusion_matrix.csv')
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score

y_val_enc = label_binarize(y_val.map({v:k for k,v in activity_map.items()}), classes=np.arange(1,7))
y_pred_proba = best['model'].predict_proba(X_val)
auc = roc_auc_score(y_val_enc, y_pred_proba, average='macro', multi_class='ovr')
print("ROC-AUC (macro, OvR) of best model:", round(auc,4))

with open('roc_auc.txt','w') as f:
    f.write(str(auc))


In [15]:
#Feature importance (if applicable
if 'RandomForest' in results:
    rf = results['RandomForest']['model']
    importances = rf.feature_importances_
    feature_names = X.columns
    fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)
    fi_df.to_csv('feature_importances.csv', index=False)
    print('Saved feature_importances.csv')

Saved feature_importances.csv


In [17]:
#Save the best model
joblib.dump(best['model'], 'best_model.joblib')
print('--- Simplified pipeline complete ---')

--- Simplified pipeline complete ---
